In [10]:
#Environment setup
!pip install great_expectations
!pip install dvc dvc-s3
from google.colab import userdata
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter
import os
import great_expectations as gx

In [3]:
#####GitHub/DVC Initialization  Configuration to Maintain Versioning --Martinez

token = userdata.get('GITHUB_TOKEN')
!git config --global user.email "marti6cp@mail.uc.edu"
!git config --global user.name "Chris Martinez"

username = "marti6cp"
repo = "BANA7075_ProjectTeam6"

!git clone https://{token}@github.com/{username}/{repo}.git

%cd /content/BANA7075_ProjectTeam6

####DVC Setup
# !dvc init
# !git add .dvc .dvcignore
# !git commit -m "Initialize DVC"



Cloning into 'BANA7075_ProjectTeam6'...
remote: Enumerating objects: 67, done.
remote: Counting objects: 100% (67/67), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 67 (delta 27), reused 63 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (67/67), 121.81 KiB | 30.45 MiB/s, done.
Resolving deltas: 100% (27/27), done.
/content/BANA7075_ProjectTeam6


In [4]:
#####KaggleHub DataIngest Module- Martinez

# Load a DataFrame with a specific version of a CSV
df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "sharmajicoder/amazon-e-commerce", "amazon_ecommerce_1M.csv",
)
#print(df.head())


####DataProcessing

df.to_csv('rawData.csv', index=False)



## Drop Null frames and establish processed data frame
df_processed = df.dropna()


#drop duplicates--- purchase date does not containenough granularity to avoid
#removing real data especially in a large ecommerce dataset drop duplicate
#functionality commented below to be turned on for real-world datasets that
#have more precise datetime information.

# df_processed.drop_duplicates(inplace=True)
# df_processed.drop_duplicates(inplace=True)

##data type consistency
##Text/string
for col in ["user_id", "product_id", "category", "subcategory", "brand",
            "seller_id", "location", "device" ,"payment_method", "delivery_status"]:
    df_processed[col] = df_processed[col].astype("string")
    df_processed[col] = df_processed[col].str.lower()


##numerics
for col in ["price" , "discount", "final_price", "rating", "review_count",
    "stock","seller_rating","shipping_time_days",
                  ]:
    df_processed[col] = pd.to_numeric(df_processed[col], errors='coerce')

## Dates
df_processed["purchase_date"] = pd.to_datetime(df_processed["purchase_date"])

print(df_processed.dtypes)



Using Colab cache for faster access to the 'amazon-e-commerce' dataset.
user_id               string[python]
product_id            string[python]
category              string[python]
subcategory           string[python]
brand                 string[python]
price                        float64
discount                     float64
final_price                  float64
rating                       float64
review_count                   int64
stock                          int64
seller_id             string[python]
seller_rating                float64
purchase_date         datetime64[ns]
shipping_time_days             int64
location              string[python]
device                string[python]
payment_method        string[python]
is_returned                     bool
delivery_status       string[python]
dtype: object


In [5]:
#Staging Data Files to be verisoned in S3 bucket
df_processed.to_csv("/content/BANA7075_ProjectTeam6/processedData.csv", index=False)


# 3. Tell DVC to track the newly saved file
!dvc add /content/BANA7075_ProjectTeam6/rawData.csv
!dvc add /content/BANA7075_ProjectTeam6/processedData.csv



# 4. Commit the DVC pointer metadata to Git
!git add /content/BANA7075_ProjectTeam6/rawData.csv.dvc #/content/BANA7075_ProjectTeam6/.gitignore
!git add /content/BANA7075_ProjectTeam6/processedData.csv.dvc #/content/BANA7075_ProjectTeam6/.gitignore


# !git commit -am "Updated Pandas raw DataFrame,Train & test via DVC"

⠋ Checking graph
Adding...:   0% 0/1 [00:00<?, ?file/s{'info': ''}]
!
          |0.00 [00:00,     ?file/s]
                                    
!
  0% |          |0/? [00:00<?,    ?files/s]
                                           
Adding rawData.csv to cache:   0% 0/1 [00:00<?, ?file/s]
Adding rawData.csv to cache:   0% 0/1 [00:00<?, ?file/s{'info': ''}]
                                                                    
  0% 0/1 [00:00<?, ?files/s]
  0% 0/1 [00:00<?, ?files/s{'info': ''}]
100% 1/1 [00:00<00:00,  5.68files/s{'info': ''}]
Adding...: 100% 1/1 [00:01<00:00,  1.33s/file{'info': ''}]

To track the changes with git, run:

	git add .gitignore rawData.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true
⠋ Checking graph
Adding...:   0% 0/1 [00:00<?, ?file/s{'info': ''}]
!
          |0.00 [00:00,     ?file/s]
                                    
!
  0% |          |0/? [00:00<?,    ?files/s]
                                           
Adding processedData.c

In [6]:
##### DVC google drive access blocked  fix via remote S3 bucket

!dvc remote add -d myremote s3://cm-education-test-bucket-201302613730-us-east-2-an/bana-storage


Setting 'myremote' as a default remote.


In [9]:
!dvc remote list
!git add .dvc/config
!git commit -m "Configure DVC default remote storage"

# Retrieve secrets from Colab
aws_key = userdata.get('AWS_ACCESS_KEY_ID')
aws_secret = userdata.get('AWS_SECRET_ACCESS_KEY')

# Set them as environment variables for AWS authentication
os.environ['AWS_ACCESS_KEY_ID'] = aws_key
os.environ['AWS_SECRET_ACCESS_KEY'] = aws_secret
!dvc push

myremote        
s3://cm-education-test-bucket-201302613730-us-east-2-an/bana-storage    
(default)
On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   .gitignore

no changes added to commit (use "git add" and/or "git commit -a")
Pushing
!
  0% |          |0/? [00:00<?,    ?files/s]
 25% 1/4 [00:00<00:02,  1.28files/s{'info': ''}]
                                                
!
  0% |          |0/? [00:00<?,    ?files/s]
                                           
Pushing to s3:   0% 0/1 [00:00<?, ?file/s]
Pushing to s3:   0% 0/1 [00:00<?, ?file/s{'info': ''}]

  0% 0.00/127M [00:00<?, ?B/s]

  0% 0.00/127M [00:00<?, ?B/s{'info': ''}]

 21% 27.2M/127M [00:00<00:02, 36.2MB/s{'info': ''}]

 61% 77.2M/127M [00:01<00:00, 79.0MB/s{'info': ''}]

100% 1

In [13]:
#######DataValidation Module######## - Martinez

###Setup GX Environment
context = gx.get_context()

###Connect to Data
# Variables
datasource_name = "ecommerce_datasource"
data_asset_name = "ecommerce_df_asset"
batch_definition_name = "big_batch"
suite_name  = "ecommerce_exp_suite"

#
#creating datasource
datasource = context.data_sources.add_pandas(
    name= datasource_name
    )
data_asset = datasource.add_dataframe_asset(
    name= data_asset_name
    )

#Create Batch Definition/Batch
batch_definition = data_asset.add_batch_definition_whole_dataframe(
    batch_definition_name
)

batch = batch_definition.get_batch(
    batch_parameters={"dataframe": df_processed}

    )

# Create Suite
suite = gx.ExpectationSuite(name = suite_name)
suite = context.suites.add(suite)

#

#### Define Expectations

##VRule to validate All training records are present
RecordMatch = gx.expectations.ExpectTableRowCountToEqual(
    value = len(df_processed)

)
#Add to suite
suite.add_expectation(RecordMatch)

##Rule to validate all Data Columnn are present and in order
ColumnMatch = gx.expectations.ExpectTableColumnsToMatchOrderedList(
    column_list=["user_id", "product_id", "category", "subcategory", "brand",
                 "price" , "discount", "final_price", "rating", "review_count",
    "stock","seller_id", "seller_rating","purchase_date","shipping_time_days",
                 "location", "device" ,"payment_method","is_returned",
                 "delivery_status"  ]
)

#add to suite
suite.add_expectation(ColumnMatch)

##Rule to validate that trainging dataframe does not contain missing data
column_list = df.columns.tolist()
for columns in column_list:       #iterates through the columns in data to check for missing data
    NoNulls = gx.expectations.ExpectColumnValuesToNotBeNull(
    column=columns
)
#add to suite
suite.add_expectation(NoNulls)

## Rule to validate Data Set still has correct nuber of columns
Column_number_check = gx.expectations.ExpectTableColumnCountToEqual(
    value=20
)

#add to suite
suite.add_expectation(Column_number_check)

## Rule to validate column as strings dType
for col in ["user_id", "product_id", "category", "subcategory", "brand",
            "seller_id", "location", "device" ,"payment_method", "delivery_status"]:
    string_check = gx.expectations.ExpectColumnValuesToBeOfType(
    column= col,
    type_= "str"
    )
    #add to suite
    suite.add_expectation(string_check)



##. Rule to validate column values are numeric dType
for col in ["price" , "discount", "final_price", "rating", "review_count",
    "stock","seller_rating","shipping_time_days",]:
    num_check = gx.expectations.ExpectColumnValuesToBeInTypeList(
    column= col,
    type_list = ["float", "int"]
    )
    suite.add_expectation(num_check)
#Add to suite





## Rule to validate column only contains dates dType
date_check = gx.expectations.ExpectColumnValuesToBeOfType(
    column= "purchase_date",
    type_= "datetime64"
    )

#add to suite
suite.add_expectation(date_check)



##Rule to validate "is_returned is bool data type
isBooleanDtype = gx.expectations.ExpectColumnValuesToBeOfType(
    column= "is_returned",
    type_= "bool"
)
#add to suite
suite.add_expectation(isBooleanDtype)


defination_name = "ecommerce validation"
validation_definition = gx.ValidationDefinition(
    data=batch_definition,
    suite = suite,
    name = defination_name
)

validation_results = validation_definition.run(batch_parameters={"dataframe": df_processed})


print(validation_results.statistics)

if validation_results.success:
  print("Data Passed Validation!")
else:
  print("Data Failed Validation !!!")
  print(validation_results.get_failed_validation_results())

INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmp5g6svpzf' for ephemeral docs site


Calculating Metrics:   0%|          | 0/9 [00:00<?, ?it/s]

{'evaluated_expectations': 24, 'successful_expectations': 24, 'unsuccessful_expectations': 0, 'success_percent': 100.0}
Data Passed Validation!
